# Exercise: Implementing and Tuning Pipeline Parallelism

Welcome to the exercise! In the demo, we saw how DeepSpeed can easily partition a simple `nn.Sequential` model. Now, it's your turn to apply these concepts to a more realistic scenario and analyze the performance trade-offs.

### Your Goal
Your mission is to:
1.  **Adapt** a non-trivial, realistic PyTorch model to make it compatible with DeepSpeed's automatic pipeline partitioning.
2.  **Implement** the baseline measurement and the DeepSpeed initialization call.
3.  **Design and run a series of experiments** to analyze the performance impact of `micro_batch_size`.
4.  **Analyze the results** to understand the concept of the "pipeline bubble" and its effect on throughput.

## 1. Environment Setup

First, let's install the necessary libraries and check our environment. This exercise requires at least two GPUs to properly evaluate pipeline parallelism.

In [1]:
# Dependencies are already installed in the repo's .venv (see ../../../../setup_env.sh).
# Uncomment when running in a fresh SageMaker / Colab environment:
# !pip install deepspeed transformers ninja


In [2]:
import os, sys, shutil, subprocess, re
import torch
import torch.nn as nn
import deepspeed
import json
from deepspeed.accelerator import get_accelerator

# DeepSpeed JIT-builds a small C++ comm op on first use; it needs `ninja` on PATH (installed in the venv).
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

print(f"PyTorch version: {torch.__version__}")
print(f"DeepSpeed version: {deepspeed.__version__}")
ACCEL = get_accelerator().device_name()   # "cuda" or "cpu"
print(f"DeepSpeed accelerator: {ACCEL}")
if torch.cuda.is_available():
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    if torch.cuda.device_count() < 2:
        print("\n!! WARNING: This exercise requires at least 2 GPUs. Performance comparison will not be meaningful otherwise. !!")
else:
    print("\n!! No GPU: the pipeline will be run with 2 CPU processes (gloo backend) on a scaled-down model."
          " Absolute throughput is not comparable to GPUs, but the micro-batch trend is still observable. !!")


[2026-08-16 15:53:48,334] [WARNING] [real_accelerator.py:199:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


PyTorch version: 2.13.0+cpu
DeepSpeed version: 0.19.5
DeepSpeed accelerator: cpu

!! No GPU: the pipeline will be run with 2 CPU processes (gloo backend) on a scaled-down model. Absolute throughput is not comparable to GPUs, but the micro-batch trend is still observable. !!


## 2. The "Realistic" Model for Our Exercise

Below is the model we will be working with. Notice that the `transformer_blocks` are stored in an `nn.ModuleList`. This is a common PyTorch pattern, but it poses a challenge for DeepSpeed's `"uniform"` partitioner, which often treats the entire `ModuleList` as a single, indivisible block.

In [3]:
# This is a mock transformer block for demonstration purposes.
class MockTransformerBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.layer1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, hidden_size)
    
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

class RealisticModel(nn.Module):
    def __init__(self, hidden_size=2048, num_layers=30, vocab_size=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.transformer_blocks = nn.ModuleList(
            [MockTransformerBlock(hidden_size) for _ in range(num_layers)]
        )
        self.output_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        for block in self.transformer_blocks:
            x = block(x)
        x = self.output_head(x)
        return x

print("Model structure defined.")

Model structure defined.


## 3. Your Task: Create the Analysis Script

Now, you will create the Python script that DeepSpeed will launch. This involves three key implementation tasks, marked with `TODO` comments inside the script.

In [4]:
%%writefile exercise_starter.py
import torch
import torch.nn as nn
import deepspeed
from deepspeed.pipe import PipelineModule
from deepspeed.accelerator import get_accelerator
import argparse
import time
import os

# --- Model Definition ---
class MockTransformerBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.layer1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, hidden_size)
    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

class RealisticModel(nn.Module):
    def __init__(self, hidden_size=2048, num_layers=30, vocab_size=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)

        # ## STUDENT TASK 1: Fix the Model Architecture ##
        # nn.ModuleList has no forward() and is treated by DeepSpeed's "uniform" partitioner as ONE opaque
        # block, so it cannot be split. nn.Sequential exposes the blocks as an ordered, callable sequence
        # of layers, which the partitioner can cut anywhere between two blocks.
        self.transformer_blocks = nn.Sequential(
            *[MockTransformerBlock(hidden_size) for _ in range(num_layers)]
        )

        self.output_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        # nn.Sequential is callable: it applies every block in order
        x = self.transformer_blocks(x)
        x = self.output_head(x)
        return x

    def to_layers(self):
        """Flat list of layers for DeepSpeed's PipelineModule (each entry can live on a different stage)."""
        return [self.embedding, *self.transformer_blocks, self.output_head]

# --- Helper Function for Performance Measurement ---
def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def measure_throughput(model, dummy_input, iterations):
    # Warm-up
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_input)
    _sync()

    start_time = time.time()
    for _ in range(iterations):
        with torch.no_grad():
            _ = model(dummy_input)
    _sync()
    end_time = time.time()

    total_samples = dummy_input.size(0) * iterations
    duration = end_time - start_time
    throughput = total_samples / duration
    return throughput

def measure_pipeline_throughput(engine, batch_fn, iterations):
    """Throughput of a DeepSpeed PipelineEngine: eval_batch() pushes one *global* batch through the
    pipeline as `train_batch_size / micro_batch_size` micro-batches (the 1F schedule)."""
    def data_iter():
        while True:
            yield batch_fn()
    for _ in range(3):                       # warm-up
        engine.eval_batch(data_iter(), compute_loss=False, reduce_output=None)
    _sync()
    torch.distributed.barrier()
    start_time = time.time()
    for _ in range(iterations):
        engine.eval_batch(data_iter(), compute_loss=False, reduce_output=None)
    _sync()
    torch.distributed.barrier()
    duration = time.time() - start_time
    return engine.train_batch_size() * iterations / duration

# --- Main Execution Logic ---
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--local_rank", type=int, default=int(os.environ.get("LOCAL_RANK", -1)))
    # Problem size knobs. Defaults reproduce the exercise on GPUs; the notebook passes smaller values on CPU.
    parser.add_argument("--hidden_size", type=int, default=2048)
    parser.add_argument("--num_layers", type=int, default=30)
    parser.add_argument("--seq_len", type=int, default=128)
    parser.add_argument("--iterations", type=int, default=20)
    parser = deepspeed.add_config_arguments(parser)
    args = parser.parse_args()

    # --- Setup ---
    global_batch_size = 64
    hidden_size = args.hidden_size
    num_layers = args.num_layers
    vocab_size = 1000
    seq_len = args.seq_len
    iterations = args.iterations
    is_rank_0 = args.local_rank <= 0
    device_name = get_accelerator().device_name()          # "cuda" or "cpu"
    first_device = f"{device_name}:0" if device_name == "cuda" else "cpu"

    # --- Baseline Measurement (Single GPU) ---
    if is_rank_0:
        print(f"\n--- Measuring Baseline Performance (Single device: {first_device}) ---", flush=True)
        # ## STUDENT TASK 2: Implement the Baseline Measurement ##
        try:
            baseline_model = RealisticModel(hidden_size, num_layers, vocab_size).to(first_device).eval()
            dummy_input = torch.randint(0, vocab_size, (global_batch_size, seq_len), device=first_device)
            baseline_throughput = measure_throughput(baseline_model, dummy_input, iterations)
            print(f"Baseline Throughput: {baseline_throughput:.2f} samples/sec", flush=True)
            del baseline_model, dummy_input
        except Exception as e:
            print(f"Could not run baseline on a single device: {e}", flush=True)
        print("--------------------------------------------------\n", flush=True)

    # --- DeepSpeed Pipeline Parallelism ---
    # deepspeed.init_distributed picks NCCL on GPU and gloo on CPU
    deepspeed.init_distributed(dist_backend="nccl" if device_name == "cuda" else "gloo")
    if torch.distributed.is_initialized():
        torch.distributed.barrier()
    world_size = torch.distributed.get_world_size()
    print(f"\n--- Rank {args.local_rank}: Setting up DeepSpeed Pipeline (world size {world_size}) ---", flush=True)

    ds_model = RealisticModel(hidden_size, num_layers, vocab_size)
    # PipelineModule cuts the flat layer list into `num_stages` contiguous chunks ("uniform" = equal
    # number of layers per stage) and places each chunk on the rank that owns that stage.
    pipe_model = PipelineModule(layers=ds_model.to_layers(), num_stages=world_size,
                                partition_method="uniform", loss_fn=None)

    # ## STUDENT TASK 3: Initialize the DeepSpeed Engine ##
    model_engine, _, _, _ = deepspeed.initialize(args=args, model=pipe_model,
                                                 model_parameters=[p for p in pipe_model.parameters() if p.requires_grad])
    model_engine.eval()

    # Input needs to be integer indices for the embedding layer. eval_batch expects (inputs, labels) tuples.
    def make_batch():
        return (torch.randint(0, vocab_size, (global_batch_size, seq_len), device=model_engine.device),
                torch.zeros(global_batch_size, dtype=torch.long, device=model_engine.device))

    pipelined_throughput = measure_pipeline_throughput(model_engine, make_batch, iterations)

    if model_engine.is_last_stage():
        print(f"\n--- Results on Last Stage (Rank {args.local_rank}) ---", flush=True)
        print(f"micro_batch_size={model_engine.train_micro_batch_size_per_gpu()}  stages={world_size}", flush=True)
        print(f"Pipelined Throughput: {pipelined_throughput:.2f} samples/sec", flush=True)
        print("------------------------------------------\n", flush=True)

if __name__ == "__main__":
    main()


Overwriting exercise_starter.py


## 4. Your Task: Set Up the Experiment Configurations

A key part of performance tuning is designing the experiments. Here, you will define the different `micro_batch_size` configurations you want to test. This will require you to think about the relationship between the global batch size, the number of GPUs (pipeline stages), and the micro-batch size.

In [5]:
import json

# The global batch size is 64, and we will use 2 GPUs (2 stages).
GLOBAL_BATCH_SIZE = 64
NUM_STAGES = 2

# ## STUDENT TASK 4: Define the Experiment Configurations ##
# Each pipeline stage sees GLOBAL_BATCH_SIZE samples per step, chopped into micro-batches of `micro_batch_size`;
# the number of micro-batches in flight is GLOBAL_BATCH_SIZE / micro_batch_size.
configs = {
    "ds_config_mbs_4.json":  { "micro_batch_size": 4,  "stages": NUM_STAGES },   # very small: 16 micro-batches, tiny kernels
    "ds_config_mbs_8.json":  { "micro_batch_size": 8,  "stages": NUM_STAGES },   # medium: 8 micro-batches
    "ds_config_mbs_16.json": { "micro_batch_size": 16, "stages": NUM_STAGES },   # larger: 4 micro-batches
    "ds_config_mbs_32.json": { "micro_batch_size": GLOBAL_BATCH_SIZE // NUM_STAGES, "stages": NUM_STAGES },  # = per-stage batch: only 2 micro-batches
}

# --- This part is complete --- #
# Base config template
base_config = {
    "train_batch_size": GLOBAL_BATCH_SIZE,
    "optimizer": { "type": "Adam", "params": { "lr": 0.001 } },
    # fp16 needs CUDA tensor cores; on CPU we stay in fp32
    "fp16": { "enabled": bool(torch.cuda.is_available()) },
    "steps_per_print": 1000,
    "wall_clock_breakdown": False,
}

for filename, pipeline_config in configs.items():
    full_config = base_config.copy()
    full_config["pipeline"] = pipeline_config
    # DeepSpeed's PipelineEngine reads the micro-batch size from this standard key
    full_config["train_micro_batch_size_per_gpu"] = pipeline_config["micro_batch_size"]
    with open(filename, 'w') as f:
        json.dump(full_config, f, indent=2)
    print(f"Created config file: {filename}")


Created config file: ds_config_mbs_4.json
Created config file: ds_config_mbs_8.json
Created config file: ds_config_mbs_16.json
Created config file: ds_config_mbs_32.json


## 5. Run the Experiments

Once you have completed all the `TODO`s above, you can run this cell to execute your experiments. It will loop through the configuration files you created and launch the DeepSpeed job for each one.

In [6]:
results = {}   # micro_batch_size -> (baseline_throughput, pipelined_throughput)

if not configs:
    print("Please complete 'STUDENT TASK 4' in the cell above before running the experiments.")
else:
    on_gpu = torch.cuda.is_available() and torch.cuda.device_count() >= NUM_STAGES
    # GPU: full-size exercise (hidden 2048, 30 layers, seq 128, 20 iterations).
    # CPU: scale the model down so each experiment finishes in ~1-2 minutes on a laptop.
    size_args = [] if on_gpu else ["--hidden_size", "512", "--num_layers", "30", "--seq_len", "32", "--iterations", "5"]
    env = dict(os.environ, DS_LOG_LEVEL="ERROR", PYTHONWARNINGS="ignore")
    if not on_gpu:
        env["OMP_NUM_THREADS"] = str(max(1, (os.cpu_count() or 2) // (2 * NUM_STAGES)))

    for config_file in configs.keys():
        print(f"\n\n{'='*60}")
        print(f"RUNNING EXPERIMENT WITH CONFIG: {config_file}")
        print(f"{'='*60}\n")
        if on_gpu:
            cmd = ["deepspeed", "--num_gpus", str(NUM_STAGES), "exercise_starter.py", "--deepspeed_config", config_file] + size_args
        else:
            # torchrun launches NUM_STAGES CPU ranks; deepspeed.init_distributed() then picks the gloo backend
            cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(NUM_STAGES),
                   "exercise_starter.py", "--deepspeed_config", config_file] + size_args
        proc = subprocess.run(cmd, env=env, text=True, capture_output=True)
        out = proc.stdout + proc.stderr
        # Show only the interesting lines (DeepSpeed is chatty)
        for line in out.splitlines():
            if re.search(r"Throughput|micro_batch_size=|Rank \d|Traceback|Error", line):
                print(line)
        b = re.search(r"Baseline Throughput: ([\d.]+)", out)
        p = re.search(r"Pipelined Throughput: ([\d.]+)", out)
        results[configs[config_file]["micro_batch_size"]] = (float(b.group(1)) if b else float("nan"),
                                                             float(p.group(1)) if p else float("nan"))
        if proc.returncode != 0:
            print(f"!! run failed with exit code {proc.returncode}; last lines:\n" + "\n".join(out.splitlines()[-15:]))
        print("\n\n")

    print("\n=== Summary ===")
    print(f"{'micro_batch_size':>16} | {'micro-batches':>13} | {'baseline (samples/s)':>20} | {'pipelined (samples/s)':>21}")
    for mbs, (b, p) in sorted(results.items()):
        print(f"{mbs:>16} | {GLOBAL_BATCH_SIZE//mbs:>13} | {b:>20.2f} | {p:>21.2f}")




RUNNING EXPERIMENT WITH CONFIG: ds_config_mbs_4.json



Baseline Throughput: 66.45 samples/sec
--- Rank 0: Setting up DeepSpeed Pipeline (world size 2) ---
--- Rank 1: Setting up DeepSpeed Pipeline (world size 2) ---
--- Results on Last Stage (Rank 1) ---
micro_batch_size=4  stages=2
Pipelined Throughput: 6.10 samples/sec





RUNNING EXPERIMENT WITH CONFIG: ds_config_mbs_8.json



Baseline Throughput: 81.07 samples/sec
--- Rank 1: Setting up DeepSpeed Pipeline (world size 2) ---
--- Rank 0: Setting up DeepSpeed Pipeline (world size 2) ---
--- Results on Last Stage (Rank 1) ---
micro_batch_size=8  stages=2
Pipelined Throughput: 12.27 samples/sec





RUNNING EXPERIMENT WITH CONFIG: ds_config_mbs_16.json



Baseline Throughput: 88.55 samples/sec
--- Rank 0: Setting up DeepSpeed Pipeline (world size 2) ---
--- Rank 1: Setting up DeepSpeed Pipeline (world size 2) ---
--- Results on Last Stage (Rank 1) ---
micro_batch_size=16  stages=2
Pipelined Throughput: 23.99 samples/sec





RUNNING EXPERIMENT WITH CONFIG: ds_config_mbs_32.json



Baseline Throughput: 89.56 samples/sec
--- Rank 1: Setting up DeepSpeed Pipeline (world size 2) ---
--- Rank 0: Setting up DeepSpeed Pipeline (world size 2) ---
--- Results on Last Stage (Rank 1) ---
micro_batch_size=32  stages=2
Pipelined Throughput: 50.03 samples/sec




=== Summary ===
micro_batch_size | micro-batches | baseline (samples/s) | pipelined (samples/s)
               4 |            16 |                66.45 |                  6.10
               8 |             8 |                81.07 |                 12.27
              16 |             4 |                88.55 |                 23.99
              32 |             2 |                89.56 |                 50.03


## 6. Analyze the Results

**Environment used for the numbers below:** no GPU available, so the pipeline was run as **2 CPU ranks** (Intel i7‑10610U, 4 physical cores → 2 threads per rank, DeepSpeed 0.19.5 CPU accelerator, gloo/shm communication) on a scaled‑down `RealisticModel` (hidden 512, 30 blocks, seq 32, global batch 64, 5 timed iterations). The single‑process baseline on the same model measured 66–90 samples/s (it uses the same 2 threads, run‑to‑run noise ±15 %). On 2 GPUs with the full‑size model (hidden 2048, seq 128) the reference numbers are ~450–465 samples/s with a much flatter curve.

| Micro Batch Size (`mbs`) | Micro‑batches per step | Pipelined Throughput (samples/sec) | Your Explanation of the Performance Trend |
|:---:|:---:|:---:|:---|
| 4   | 16 | **6.10**  | **Lowest.** With 16 tiny micro‑batches every step pays 16× the fixed per‑micro‑batch overhead: Python scheduling in the pipeline engine, a gloo `send`/`recv` of the activation tensor between the two stages, and matmuls that are far too small (4×32 tokens × 512) to use the cores efficiently. Even on GPUs the analogous cost is kernel‑launch latency and the *pipeline bubble* – the first stage idles while the last micro‑batch drains, so throughput is lower than at the optimum. |
| 8   | 8  | **12.27** | **Increasing (~2×).** Halving the number of micro‑batches halves the fixed overhead and doubles the work per matmul, so throughput almost exactly doubles: we are still entirely in the overhead‑dominated regime. On GPUs the gain from 4→8 is small (453→461 samples/s) because kernels are already reasonably efficient at mbs=4 and the bubble fraction (stages−1)/(micro‑batches+stages−1) only drops from 1/17 to 1/9. |
| 16  | 4  | **23.99** | **Still doubling.** Overhead now dominates less, matmuls (16×32×512) are big enough for BLAS to reach decent efficiency, and only 4 micro‑batches cross the stage boundary. On GPU this is typically the sweet spot (464 samples/s in the reference run): the bubble is only 1/5 of a step and each micro‑batch is large enough to saturate the SMs. |
| 32  | 2  | **50.03** | **Highest here (but ~0.6× the single‑process baseline).** With just 2 micro‑batches there is almost no pipelining left – stage 2 sits idle while stage 1 computes the first half and vice‑versa – yet on this CPU that still wins because both ranks share the same 4 cores, so there is nothing to gain from overlapping compute anyway and every avoided micro‑batch removes overhead. On real multi‑GPU hardware this is where throughput *decreases* again (461 samples/s vs 464 at mbs=16): with only 2 micro‑batches half of each step is bubble, per‑micro‑batch activation memory doubles, and the second GPU is idle 50 % of the time. |

**Take‑aways**

* The micro‑batch size trades **pipeline utilisation** (many small micro‑batches keep every stage busy and shrink the bubble, whose fraction is (S−1)/(M+S−1) for S stages and M micro‑batches) against **per‑micro‑batch efficiency and overhead** (each micro‑batch has a fixed cost: kernel launches / Python scheduling, one activation transfer per stage boundary, and small matmuls under‑utilise the hardware).
* On GPUs with real parallel devices the optimum sits in the middle (mbs ≈ 8–16 for global batch 64 and 2 stages). On this CPU‑only simulation the two ranks compete for the same cores, so parallelism can never pay off and the curve is monotonic in mbs — a good reminder that pipeline parallelism only helps when the stages run on genuinely separate compute.
* Practical rule: choose the largest micro‑batch that still leaves ≥ 4·(S−1) micro‑batches per step (bubble ≤ ~20 %) and fits in memory.
